In [ ]:
!pip install transformers sentencepiece accelerate sacremoses nltk -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 16.2 MB/s eta 0:00:00


In [ ]:
import re
import random
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet as wn

SEED = 42
random.seed(SEED)

INPUT_CSV = "/content/drive/MyDrive/ASQP/data_processado/train_processed.csv"

TEXT_COL = "text"
TARGET_COL = "target"
INPUT_COL = "input"

MAX_ATTEMPTS = 30
N_AUG_PER_REVIEW = 2

In [ ]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

STOP_WORDS_PT = set(stopwords.words("portuguese"))

OUT_CSV = "/content/drive/MyDrive/ASQP/saida_asqp/aug_sr.csv"
AUDIT_CSV = "/content/drive/MyDrive/ASQP/saida_asqp/audit_aug_sr.csv"

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
df = pd.read_csv(INPUT_CSV).reset_index(drop=True)

print("Tamanho original:", len(df))
print(df.columns.tolist())

display(df.head())

Tamanho original: 730
['id', 'text', 'input', 'target', 'quadruples', 'augmented', 'augmentation', 'source_id']


,id,text,input,target,quadruples,augmented,augmentation,source_id
0,ex_00000,hotel com condições gerais muito más infraestr...,extrair quadruplas:hotel com condições gerais ...,[A] hotel [C] general [O] más [P] neg [SSEP] [...,"[{'category': 'general', 'aspect': {'term': 'H...",False,none,ex_00000
1,ex_00001,o hotel segue corretamente os padrões do mercu...,extrair quadruplas:o hotel segue corretamente ...,[A] localização [C] location [O] excelente [P]...,"[{'category': 'location', 'aspect': {'term': '...",False,none,ex_00001
2,ex_00002,o hotel é muito luxuoso e em frente o central ...,extrair quadruplas:o hotel é muito luxuoso e e...,[A] hotel [C] general [O] luxuoso [P] pos [SSE...,"[{'category': 'general', 'aspect': {'term': 'h...",False,none,ex_00002
3,ex_00003,o hotel em geral é bom o que achamos ruim foi ...,extrair quadruplas:o hotel em geral é bom o qu...,[A] hotel [C] general [O] bom [P] pos [SSEP] [...,"[{'category': 'general', 'aspect': {'term': 'h...",False,none,ex_00003
4,ex_00004,em cidades grandes e com muitas atrações como ...,extrair quadruplas:em cidades grandes e com mu...,[A] hotel [C] location [O] próximo do louvre d...,"[{'category': 'location', 'aspect': {'term': '...",False,none,ex_00004


In [ ]:
def clean_text(text):
  text = str(text)
  text = re.sub(r"\s+", " ", text)
  return text.strip()


def parse_target_quads(target):
  target = str(target)

  parts = [
      p.strip()
      for p in target.split("[SSEP]")
      if p.strip()
  ]

  quads = []

  pattern = (
        r"\[A\]\s*(.*?)\s*"
        r"\[C\]\s*(.*?)\s*"
        r"\[O\]\s*(.*?)\s*"
        r"\[P\]\s*(.*)"
  )

  for part in parts:
    m = re.search(pattern, part, flags=re.IGNORECASE)

    if not m:
      continue

    quads.append({
            "aspect": m.group(1).strip(),
            "category": m.group(2).strip().lower(),
            "opinion": m.group(3).strip(),
            "polarity": m.group(4).strip().lower()
    })

  return quads


def get_critical_terms(target):

  quads = parse_target_quads(target)
  terms = []

  for q in quads:
    for term in [q["aspect"], q["opinion"]]:
      term = str(term).strip()

      if term and term.lower() not in ["null", "none", "nan", "_"]:
        terms.append(term)

  return sorted(set(terms), key=len, reverse=True)

def get_terms_present_in_text(text, target):
  text_low = clean_text(text).lower()

  return [
        term for term in get_critical_terms(target)
        if term.lower() in text_low
    ]


def missing_terms(text, terms):
  text_low = clean_text(text).lower()

  return [
        term for term in terms
        if term.lower() not in text_low]

In [ ]:
def mask_critical_terms(text, target):
  masked_text = clean_text(text)
  terms = get_terms_present_in_text(masked_text, target)

  mapping = {}

  for i, term in enumerate(terms):
      placeholder = f"ASQPTERM{i}X"

      pattern = rf"(?<!\w){re.escape(term)}(?!\w)"

      masked_text = re.sub(
            pattern,
            placeholder,
            masked_text,
            flags=re.IGNORECASE
      )

      if placeholder in masked_text:
          mapping[placeholder] = term

  return masked_text, mapping, list(mapping.values())


def restore_critical_terms(text, mapping):
  restored = str(text)

  for placeholder, original in sorted(mapping.items(),key=lambda x: len(x[0]),reverse=True):
      pattern = rf"(?<!\w){re.escape(placeholder)}(?!\w)"

      restored = re.sub(
            pattern,
            original,
            restored,
            flags=re.IGNORECASE)

  return clean_text(restored)

In [ ]:
row = df.sample(n=1, random_state=42).iloc[0]

masked, mapping, terms = mask_critical_terms(row[TEXT_COL], row[TARGET_COL])

print("termos:", terms)
print("\nmascarado:")
print(masked)
print("\nmapping:")
print(mapping)

restored = restore_critical_terms(masked, mapping)

print("\nrestaurado:")
print(restored)

print("\nFaltando após restaurar:")
print(missing_terms(restored, terms))

termos: ['restaurantes', 'atendimento', 'exelentes', 'shopping', 'cassino', 'piscina', 'animada', 'melhore', 'caesars', 'adorei', 'melhor', 'shows', 'ótimo']

mascarado:
estive em vegas no ASQPTERM8X e ASQPTERM9X ótima localizaçao diria a ASQPTERM10X da cidade perto de todos os grandes hotéis com o ASQPTERM10X ASQPTERM4X na minha opiniao ASQPTERM2X ASQPTERM0X os ASQPTERM7X ASQPTERM11X com isso nao é preciso a locaçao de carro o ASQPTERM10X ASQPTERM3X esta no hotel ASQPTERM5X ASQPTERM6X e um ASQPTERM12X ASQPTERM1X podem ficar sem medo é uma das melhores opçoes levando em conta o custo benefício boa viagem

mapping:
{'ASQPTERM0X': 'restaurantes', 'ASQPTERM1X': 'atendimento', 'ASQPTERM2X': 'exelentes', 'ASQPTERM3X': 'shopping', 'ASQPTERM4X': 'cassino', 'ASQPTERM5X': 'piscina', 'ASQPTERM6X': 'animada', 'ASQPTERM7X': 'melhore', 'ASQPTERM8X': 'caesars', 'ASQPTERM9X': 'adorei', 'ASQPTERM10X': 'melhor', 'ASQPTERM11X': 'shows', 'ASQPTERM12X': 'ótimo'}

restaurado:
estive em vegas no caesars e a

In [ ]:
def obter_sinonimos_pt(word):
  word = word.lower().strip(".,!?;:()[]{}\"'")
  synonyms = set()

  for synset in wn.synsets(word, lang="por"):
      for lemma in synset.lemmas(lang="por"):
        s = lemma.name().replace("_", " ").lower().strip()

        if s != word:
          synonyms.add(s)

  return list(synonyms)


def is_mask_token(word):
  return bool(re.search(r"\[TERM\d+\]",word,flags=re.IGNORECASE))


def clean_token(word):
  return word.lower().strip(".,!?;:()[]{}\"'")


def preserve_punctuation(original, new):
  prefix = re.match(r"^\W+", original)
  suffix = re.search(r"\W+$", original)

  prefix = prefix.group(0) if prefix else ""
  suffix = suffix.group(0) if suffix else ""

  return f"{prefix}{new}{suffix}"


def is_eligible_for_sr(word):
  token = clean_token(word)

  if not token:
    return False

  if is_mask_token(word):
    return False

  if token in STOP_WORDS_PT:
    return False

  if token.isdigit():
    return False

  if len(token) <= 2:
    return False

  return True

In [ ]:
def synonym_replacement_once(text, target):
  masked_text, mapping, protected_terms = mask_critical_terms(text,target)

  words = masked_text.split()
  new_words = words.copy()

  n = max(1, len(words) // 3)

  eligible = [
      i for i, w in enumerate(words)
      if is_eligible_for_sr(w)
  ]

  random.shuffle(eligible)

  changed = 0

  for idx in eligible:
    if changed >= n:
      break

    word = new_words[idx]
    token = clean_token(word)

    synonyms = obter_sinonimos_pt(token)

    if not synonyms:
      continue

    synonym = random.choice(synonyms)
    new_words[idx] = preserve_punctuation(word, synonym)

    changed += 1

  augmented = " ".join(new_words)
  augmented = restore_critical_terms(augmented, mapping)

  return clean_text(augmented), protected_terms, changed


def synonym_replacement_safe(text, target, max_attempts=5):
  original = clean_text(text)

  for attempt in range(1, max_attempts + 1):
    try:
      augmented, protected_terms, changed = synonym_replacement_once(original,target)

      missing = missing_terms(augmented,protected_terms)

      if len(missing) == 0 and changed > 0:
        return {
                    "text": augmented,
                    "ok": True,
                    "attempts": attempt,
                    "protected_terms": protected_terms,
                    "missing_terms": [],
                    "changed_words": changed
                }

    except Exception as e:
            missing = [f"ERRO: {e}"]

  return {
      "text": original,
      "ok": False,
      "attempts": max_attempts,
      "protected_terms": get_terms_present_in_text(original, target),
      "missing_terms": missing,
      "changed_words": 0
  }

In [ ]:
for i in range(10):
  row = df.sample(n=1, random_state=SEED + i).iloc[0]

  result = synonym_replacement_safe(
        row[TEXT_COL],
        row[TARGET_COL],
        max_attempts=5
  )

  print(
        f"{i+1:02d} | "
        f"preservado={result['ok']} | "
        f"tentativas={result['attempts']} | "
        f"protegidos={len(result['protected_terms'])} | "
        f"alteradas={result['changed_words']} | "
        f"faltando={len(result['missing_terms'])}"
    )

01 | preservado=True | tentativas=1 | protegidos=13 | alteradas=13 | faltando=0
02 | preservado=True | tentativas=1 | protegidos=9 | alteradas=5 | faltando=0
03 | preservado=True | tentativas=1 | protegidos=11 | alteradas=7 | faltando=0
04 | preservado=True | tentativas=1 | protegidos=17 | alteradas=10 | faltando=0
05 | preservado=True | tentativas=1 | protegidos=4 | alteradas=18 | faltando=0
06 | preservado=True | tentativas=1 | protegidos=7 | alteradas=16 | faltando=0
07 | preservado=True | tentativas=1 | protegidos=13 | alteradas=12 | faltando=0
08 | preservado=True | tentativas=1 | protegidos=2 | alteradas=2 | faltando=0
09 | preservado=True | tentativas=1 | protegidos=8 | alteradas=15 | faltando=0
10 | preservado=True | tentativas=1 | protegidos=6 | alteradas=15 | faltando=0


In [ ]:
rows = []
audit_rows = []

for idx, row in df.iterrows():
  original_row = row.to_dict()

  original_row["augmented"] = False
  original_row["augmentation"] = "none"
  original_row["source_id"] = row.get("id", idx)
  original_row["critical_terms_preserved"] = True
  original_row["sr_attempts"] = 0
  original_row["changed_words"] = 0
  original_row["protected_terms"] = str(get_terms_present_in_text(row[TEXT_COL],row[TARGET_COL]))
  original_row["missing_terms"] = "[]"

  rows.append(original_row)


for idx, row in tqdm(
  df.iterrows(),
  total=len(df),
  desc="Gerando 2 SR por review"
):
  for aug_number in range(1, N_AUG_PER_REVIEW + 1):

      result = synonym_replacement_safe(
            row[TEXT_COL],
            row[TARGET_COL],
            max_attempts=5)

      new_row = row.to_dict()
      new_row[TEXT_COL] = result["text"]

      if INPUT_COL in new_row:
        new_row[INPUT_COL] = (
                "extrair quádruplas: "
                + clean_text(result["text"])
        )

      new_row["augmented"] = True
      new_row["augmentation"] = "SR"
      new_row["source_id"] = row.get("id", idx)
      new_row["aug_number"] = aug_number
      new_row["critical_terms_preserved"] = result["ok"]
      new_row["sr_attempts"] = result["attempts"]
      new_row["changed_words"] = result["changed_words"]
      new_row["protected_terms"] = str(result["protected_terms"])
      new_row["missing_terms"] = str(result["missing_terms"])

      rows.append(new_row)

      audit_rows.append({
            "source_id": row.get("id", idx),
            "aug_number": aug_number,
            "critical_terms_preserved": result["ok"],
            "sr_attempts": result["attempts"],
            "changed_words": result["changed_words"],
            "protected_terms": str(result["protected_terms"]),
            "missing_terms": str(result["missing_terms"]),
            "original_text": row[TEXT_COL],
            "augmented_text": result["text"],
            "target": row[TARGET_COL]
      })


aug_df = pd.DataFrame(rows)

aug_df = aug_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

audit_df = pd.DataFrame(audit_rows)

Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)

aug_df.to_csv(OUT_CSV, index=False)
audit_df.to_csv(AUDIT_CSV, index=False)

print("Original:", len(df))
print("Final:", len(aug_df))
print("Esperado:", len(df) * 3)
print("Dataset salvo em:", OUT_CSV)
print("Auditoria salva em:", AUDIT_CSV)

Gerando 2 SR por review:   0%|          | 0/730 [00:00<?, ?it/s]

Original: 730
Final: 2190
Esperado: 2190
Dataset salvo em: /content/drive/MyDrive/ASQP/saida_asqp/aug_sr.csv
Auditoria salva em: /content/drive/MyDrive/ASQP/saida_asqp/audit_aug_sr.csv


In [ ]:
print("Total de aumentadas:", aug_df["augmented"].sum())
print("Falhas de preservação:", (~aug_df["critical_terms_preserved"]).sum())

display(aug_df[aug_df["critical_terms_preserved"] == False]
 [
        [
            "id",
            "source_id",
            "augmentation",
            "sr_attempts",
            "changed_words",
            "protected_terms",
            "missing_terms",
            TEXT_COL,
            TARGET_COL
        ]
    ].head(30)
)

Total de aumentadas: 1460
Falhas de preservação: 2


,id,source_id,augmentation,sr_attempts,changed_words,protected_terms,missing_terms,text,target
1753,ex_00321,ex_00321,SR,5,0,"['café da manhã', 'imigrantes', 'péssimo', 'ba...",[],péssimo staff fraco café da manhã bairro de im...,[A] staff [C] service [O] péssimo [P] neg [SSE...
1989,ex_00321,ex_00321,SR,5,0,"['café da manhã', 'imigrantes', 'péssimo', 'ba...",[],péssimo staff fraco café da manhã bairro de im...,[A] staff [C] service [O] péssimo [P] neg [SSE...


In [ ]:
import difflib

audit = pd.read_csv(AUDIT_CSV)

for _, row in audit.head(10).iterrows():
  original = str(row["original_text"]).split()
  augmented = str(row["augmented_text"]).split()

  matcher = difflib.SequenceMatcher(None, original, augmented)

  print("=" * 100)

  for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == "replace":
      print(
      f"{' '.join(original[i1:i2])}"
      f"  -->  "
      f"{' '.join(augmented[j1:j2])}"
      )

bastante  -->  assaz
quartos  -->  apartamento
piso  -->  andar
sanita  -->  casa de banho
ponta  -->  pico
corredor  -->  viela
ponta  -->  ponteiro
equipamentos  -->  ferramenta
bastante  -->  abundante
pouco  -->  pingo
quarto  -->  quinto
privacidade  -->  clandestinidade
vez  -->  ocasião
piso todo sempre  -->  assoalho unidade incessantemente
alguém  -->  pessoa
corredor  -->  passagem
internet  -->  ciberespaço
recepção  -->  recebimento
recepção  -->  recibo
sala  -->  antessala
escuro  -->  soturno
mínimo  -->  minimum
junto  -->  contíguo
metro  -->  trem de m
quarto  -->  aposento
bom  -->  boa centrais de
outro  -->  díspar
bastante  -->  muita
quartos  -->  apartamento
piso  -->  pavimento
sanita  -->  casa de banho
ponta  -->  pico
corredor  -->  sala
ponta  -->  pico
equipamentos  -->  equipamento
bastante  -->  copioso
pouco  -->  um tanto
quarto  -->  sala
privacidade  -->  clandestinidade
vez  -->  jogo
piso todo sempre  -->  pavimento unidade continuamente
alguém  --